# 6.5. Custom Layers
D2L의 Custom Layers장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. Custom Layer란?

PyTorch에는 이미 다양한 Layer들이 존재한다.

예를 들어서
- `nn.Linear`
- `nn.ReLU`
- `nn.Conv2d`
- `nn.Dropout`

등이 있다.

하지만 연구나 새로운 모델을 구현하다 보면 PyTorch에 없는 특별한 연산이 필요할 수도 있다. 이럴 때 `nn.Module`을 상속해 직접 Layer를 만들 수 있다.

기본 구조는 이렇다.

1. `nn.Module` 상속
2. `__init__()`에서 필요한 구성요소 정의
3. `forward()`에서 실제 연산 정의

## 2. Custom Layer의 기본 구조

In [2]:
from torch import nn
import torch.nn.functional as F

Custom Layer 역시 하나의 `nn.Module`이다.

따라서 기본적인 형태는 다음과 같다.

In [3]:
class MyLayer(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, X):
        return X

    입력 X -> forward() -> 출력

layer(X) 를 실행하면 내부적으로 layer.forward(X) 연산이 수행된다.

## 3. Parameter가 없는 Custom Layer

`CenteredLayer`는 입력 전체의 평균을 빼서 출력의 평균을 0 근처로 만드는 단순한 layer이다.

모든 Layer가 반드시 Weight와 Bias를 가져야 하는 것은 아니다. 단순히 어떤 연산만 수행하는 Layer도 만들 수 있다. 예를 들어 입력 데이터의 평균을 빼는 Layer를 만들어보자

$$
Y = X - \text{mean}(X)
$$

이 연산을 수행하면 출력 데이터의 평균은 거의 0이 된다.

In [4]:
class CenteredLayer(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, X):
        return X - X.mean()

여기에는

self.weight, self.bias 같은게 없다. 학습해야할 parameter도 없다. 그냥 입력을 받아서 X - X.mean() 연산만 수행한다.

## 4. CenteredLayer 사용하기

In [5]:
layer = CenteredLayer()

X = torch.tensor([1., 2., 3., 4., 5.])

Y = layer(X)

print(Y)

tensor([-2., -1.,  0.,  1.,  2.])


입력이 1,2,3,4,5면 평균은 3이다. 그래서 모든 값에서 3을 빼면
    [-2, -1, 0, 1, 2]

가 된다. 평균은 0이 된다.

## 5. 만든 Layer를 모델에 넣기

Custom Layer라고 해서 특별하게 사용할 필요는 없다.

nn.Linear, nn.ReLU와 똑같은 nn.Module이므로 nn.Sequential 안에도 넣을 수 있다. D2L도 custom layer가 더 복잡한 model의 구성 요소로 들어갈 수 있음을 보여준다.

In [ ]:
net = nn.Sequential(
    nn.Linear(8, 128),
    CenteredLayer()
)

X = torch.rand(4, 8) # [batch size, feature 수]

Y = net(X)

print(Y.shape)
print(Y.mean())

torch.Size([4, 128])
tensor(2.3283e-09, grad_fn=<MeanBackward0>)


입력 shape = [4, 8]

`nn.Linear(8, 128)`을 통과하면

[4, 128]이 된다.

이후에 `CenteredLayer`가 전체 평균을 빼기 때문에 출력의 평균은 거의 0이 된다.

## 6. Parameter가 있는 Custom Layer

이번에는 학습 가능한 Parameter를 가진 Layer를 직접 만들어 보겠다. PyTorch에서 `nn.Parameter`를 사용하면 Tensor를 모델이 학습해야 하는 Parameter로 등록할 수 있다.

예를 들어

```py
self.weight = nn.Parameter(~~) # 이렇게 만들면 

# PyTorch가 자동으로 안에 넣어준다.
model.parameters()
# 그러면 optimizer도 이 값을 찾아서 업데이트할 수 있다.
```

## 7. 직접 Linear Layer 만들기

우리가 알고 있는 `nn.Linear`와 비슷한 것을 만들어 보자.

Linear는 
$$
Y = XW + b
$$

여기에 ReLU까지 적용하면
$$
Y = ReLU(XW + b)
$$

In [7]:
class MyLinear(nn.Module):

    def __init__(self, in_features, out_features):
        super().__init__()

        self.weight = nn.Parameter(
            torch.randn(in_features, out_features)
        )

        self.bias = nn.Parameter(
            torch.zeros(out_features)
        )

    def forward(self, X):

        linear = X @ self.weight + self.bias

        return F.relu(linear)

## 8. Weight와 Bias의 Shape 이해하기

예를 들어서

    layer = MyLinear(5, 3)
이라고 했을 때 입력 feature는 5개, 출력 feature는 3개라는 뜻이다. 

우리가 만든 구현에서는

```text
X.shape      = [batch_size, 5]
weight.shape = [5, 3]
bias.shape   = [3]
```

## 9. nn.Linear와 Weight Shape가 다른 이유

우리가 방금 만든 MyLinear는 

    weight.shape = [in_features, out_features]

그런데 실제 PyTorch의 

    nn.Linear(5, 3)
    weight.shape == [3, 5]

왜냐하면 `nn.Linear` 내부는 개념적으로
$$
XW^T + b
$$

형태를 사용하기 때문이다.

In [8]:
linear = nn.Linear(5, 3)

print(linear.weight.shape)
print(linear.bias.shape)

torch.Size([3, 5])
torch.Size([3])


In [9]:
my_linear = MyLinear(5, 3)

print(my_linear.weight.shape)
print(my_linear.bias.shape)

torch.Size([5, 3])
torch.Size([3])


## 10. Forward 과정 확인

In [10]:
layer = MyLinear(5, 3)

X = torch.randn(2, 5)

Y = layer(X)

print("X shape:", X.shape)
print("Weight shape:", layer.weight.shape)
print("Bias shape:", layer.bias.shape)
print("Y shape:", Y.shape)

X shape: torch.Size([2, 5])
Weight shape: torch.Size([5, 3])
Bias shape: torch.Size([3])
Y shape: torch.Size([2, 3])


```text
X
[2, 5]

↓

X @ W
[2, 5] @ [5, 3]

↓

[2, 3]

↓

Bias 추가

↓

ReLU

↓

출력
[2, 3]
```

## 11. Parameter가 정말 등록되는지 확인하기

In [11]:
layer = MyLinear(5, 3)

for name, param in layer.named_parameters():
    print(name, param.shape)

weight torch.Size([5, 3])
bias torch.Size([3])


self.weight = nn.Parameter(~~~)
self.bias = nn.Parameter(~~~)

로 정의했기 때문에 자동으로 등록된 걸 볼 수 있다.

PyTorch가 자동으로 학습 Parameter로 등록했기 때문에 Optimizer가 찾아서 gradient를 이용해 업데이트할 수 있다.

## 12. 실제로 Gradient가 계산되는지 확인하기

In [ ]:
layer = MyLinear(5, 3)

X = torch.randn(2, 5)

Y = layer(X)

loss = Y.sum()

loss.backward() # 기존 신경망 Layer같이 역전파 가능

print(layer.weight.grad)
print(layer.bias.grad)

tensor([[-0.5182, -0.3138, -0.3138],
        [ 1.0303,  0.0345,  0.0345],
        [-1.0705,  1.7849,  1.7849],
        [ 0.8465, -0.7594, -0.7594],
        [ 1.3465, -1.2326, -1.2326]])
tensor([1., 1., 1.])


## 13. Custom Layer 여러 개 연결하기

In [13]:
net = nn.Sequential(
    MyLinear(64, 8),
    MyLinear(8, 1)
)

X = torch.randn(2, 64)

Y = net(X)

print(Y.shape)

torch.Size([2, 1])


## 14. 언제 Custom Layer를 쓸까?

기존 PyTorch Layer만으로 원하는 연산을 표현하기 어렵다면
직접 Layer를 만들 수 있다.

예를 들어

- 특별한 수학 연산
- 새로운 활성화 방식
- 여러 연산을 하나의 Layer로 묶기
- 논문에서 제안된 새로운 구조 구현
- 특정 도메인에 특화된 연산

등에 사용할 수 있다.

Custom Layer도 `nn.Module`이므로 기존 PyTorch Layer와 동일하게 모델 안에서 사용할 수 있다.

만약 논문 구현을 한다면 이런 패턴을 자주 볼 수도 있다.
```py
class SomethingLayer(nn.Module):

    def __init__(self, ...):
        super().__init__()

        # Layer / Parameter 정의

    def forward(self, X):

        # 논문에서 정의한 연산

        return X
```

새로운 계산식이 나왔다고 PyTorch 자체 수정이 아니라 새로운 `nn.Module`을 만들어 구현하면 된다.

## 15. 오늘의 정리

- PyTorch에 없는 Layer는 `nn.Module`을 상속해서 직접 만들 수 있다.
- `__init__()`에서는 Layer가 사용할 구성요소와 Parameter를 정의한다.
- `forward()`에서는 입력이 들어왔을 때 수행할 실제 계산을 정의한다.
- Weight나 Bias가 없는 단순한 Custom Layer도 만들 수 있다.
- 학습할 값은 `nn.Parameter`로 정의할 수 있다.
- `nn.Parameter`는 자동으로 모델의 학습 Parameter로 등록된다.
- Custom Layer도 `nn.Sequential` 안에 일반 Layer처럼 넣을 수 있다.
- Custom Layer에서도 autograd가 동작하므로 Weight와 Bias의 gradient가 자동으로 계산된다.
- `X @ W + b`에서 Tensor shape가 어떻게 변하는지 이해하는 것이 중요하다.
- 직접 만든 `MyLinear`와 PyTorch `nn.Linear`는 Weight를 저장하는 방향이 다를 수 있다.